In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

gobp = pd.read_csv("../DPSC_CNCC/GOBP_AccelEvol_Input.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("../DPSC_CNCC/HPO_AccelEvol_Input.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
    

In [ ]:
pos = d_BP['Positive Regulation Of DNA-templated Transcription (GO:0045893)']
neg = d_BP['Negative Regulation Of DNA-templated Transcription (GO:0045892)']
a = d_BP['Regulation Of DNA-templated Transcription (GO:0006355)']

In [ ]:
blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
            "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
            "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]

#Genes that are near or in IGV etc. areas
blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP"]

#Genes that seem a little iffy since they are embedded in regions with lots of the same gene
iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
orth = orth[~orth["Gene name"].isin(blacklist + blacklist2)]

olfs = []
hlas = []
ugt = []
for i in orth["Gene name"]:
    try:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)
    except:
        pass
orth = orth[~orth["Gene name"].isin(olfs + hlas + ugt)]
pd.DataFrame(orth.drop_duplicates("Gene name")["Gene name"]).to_csv("Whitelist_Genes.txt", sep = "\t", header = False, index = False)
pd.DataFrame(orth.drop_duplicates("Gene name")["Gene name"])

In [ ]:
def filter_nc(file):
    v = pd.read_csv(file, sep = "\t")
    #Genes that are directly adjacent to centromeres
    blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
                "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
                "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]

    #Genes that are near or in IGV etc. areas
    blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP"]
    
    #Genes that seem a little iffy since they are embedded in regions with lots of the same gene
    iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
    orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
    orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
    v = v[v["NearestGene"].isin(orth["Gene name"])]
    v = v[~v["NearestGene"].isin(blacklist + blacklist2)]
    
    olfs = []
    hlas = []
    ugt = []
    for i in v["NearestGene"]:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)
    
    #Remove olfactory receptors and HLA genes
    v = v[~v["NearestGene"].isin(olfs + hlas + ugt)]
    
    #Filter to only genes with sufficient nearby substituions
    v = v[(v["Species1 Sum Total_Vars"] > 100) | (v["Species2 Sum Total_Vars"] > 100)]

    #Recompute FDR after filtering
    v["FDR Difference"] = fdrcorrection(v["p-value Difference"])[1]
    v["FDR Difference Corr Tot"] = fdrcorrection(v["p-value Difference Corr Tot"])[1]
    v["FDR Difference Corr S"] = fdrcorrection(v["p-value Difference Corr S"])[1]
    
    return v.sort_values("p-value Difference Corr Tot")

In [ ]:
def mvp(v, xlab, title, stat = "PhyloP Difference", fdr_col = "FDR", x_col = "PhyloP Difference"):
    plt.figure(figsize=(6, 4), dpi=600)
    sig = []
    #Also filtered out sites that were variable in apes or variable across human populations
    for index, row in v.iterrows():
        if row[fdr_col] < 0.05 and row[stat] < 0:
            sig.append("Chimp accelerated")
        elif row[fdr_col] < 0.05 and row[stat] > 0:
            sig.append("Human accelerated")
        else:
            sig.append("Not significant")
    v["Category"] = sig
    v["-Log$_{10}$(FDR)"] = -np.log10(v[fdr_col])
    v = v.sort_values("Category")
    sns.scatterplot(data = v, x = stat, y = "-Log$_{10}$(FDR)", hue = "Category", palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey"})

    xlim1 = min(v[stat]) - abs(floor(0.1*min(v[stat])))
    xlim2 = max(v[stat]) + abs(floor(0.1*max(v[stat])))
    plt.xlim(xlim1, xlim2)
    plt.hlines(-np.log10(0.05), xlim1, xlim2, linestyles="dashed", colors="k")
    plt.legend(bbox_to_anchor = (1, 1))
    plt.xlabel(xlab, size = 13)
    plt.ylabel("-Log$_{10}$(FDR)", size = 13)
    plt.xticks(size = 12)
    plt.yticks(size = 12)
    plt.title(title, size = 14)
    plt.legend([], [], frameon = False)
    plt.show()

In [ ]:
v = filter_nc("PhyloP_NC_NoFiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_Results.txt")
v2 = filter_nc("PhyloP_NC_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_Results.txt")

sig = np.intersect1d(v[(v["FDR Difference"] < 0.05) & (v["FDR Difference Corr Tot"] < 0.05)]["NearestGene"], v2[(v2["FDR Difference"] < 0.25) & (v2["FDR Difference Corr Tot"] < 0.25)]["NearestGene"])
nsig = np.intersect1d(v[(v["FDR Difference"] > 0.05) & (v["FDR Difference Corr Tot"] > 0.05)]["NearestGene"], v2[(v2["FDR Difference"] > 0.05) & (v2["FDR Difference Corr Tot"] > 0.05)]["NearestGene"])
v_sig = v[v["NearestGene"].isin(sig)]
v_nsig = v[v["NearestGene"].isin(nsig)]
v2_sig = v2[v2["NearestGene"].isin(sig)]
v2_nsig = v2[v2["NearestGene"].isin(nsig)]

v_filt = pd.concat([v_sig, v_nsig])
v2_filt = pd.concat([v2_sig, v2_nsig])
#mvp(v_filt, xlab = "Difference in PhyloP", title = "Accelerated evolution of $\it{cis}$-regulatory neighborhoods", fdr_col = "FDR Difference")

In [ ]:
v_filt = v_filt.drop(["PhyloP L2FC", "Z-score Difference", "Z-score Difference Corr Tot", "Z-score Difference Corr S", "Z-score L2FC", "p-value L2FC", "FDR L2FC"], axis = 1)
v_filt.columns = ["Gene symbol", "Human PhyloP-weighted sum", "Number of human-derived substitutions", "Chimp PhyloP-weighted sum", "Number of chimp-derived substitutions", "Difference in PhyloP-weighted sum", "p-value", "Total corrected p-value", "PhyloP distribution difference p-value", "FDR", "Total corrected FDR", "PhyloP distribution difference FDR"]
k = []
for index, row in v_filt.iterrows():
    if row["Difference in PhyloP-weighted sum"] < 0 and row["FDR"] < 0.05 and row["Total corrected FDR"] < 0.05:
        k.append("Chimpanzee-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] < 0 and row["p-value"] < 0.05 and row["Total corrected p-value"] < 0.05:
        k.append("Nominally chimpanzee-accelerated, p < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["FDR"] < 0.05 and row["Total corrected FDR"] < 0.05:
        k.append("Human-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["p-value"] < 0.05 and row["Total corrected p-value"] < 0.05:
        k.append("Nominally human-accelerated, p < 0.05")
    else:
        k.append("Not accelerated")
v_filt["Acceleration"] = k
v_filt.to_csv("Supplemental_Tables/Supplemental_Table5.csv", index = False)

In [ ]:
binomtest(29, 13 + 29)

In [ ]:
har = pd.read_csv("Girskis_HARs.csv")

In [ ]:
har_genes = []
HAR_IDs = []
for index, row in har.iterrows():
    har_genes = har_genes + row["Nearest Neighboring Gene"].split(";")
    for j in row["Nearest Neighboring Gene"].split(";"):
        if j in list(v_sig["NearestGene"]):
            HAR_IDs.append(row["HAR_ID"])

In [ ]:
haq = pd.read_csv("../PosSelect_ForPub/haqer.hg38.bed", sep = "\t", header = None)
haq_pos = []

for index, row in haq.iterrows():
    for i in range(row[1], row[2]):
        haq_pos.append([row[0] + ":" + str(i), row[3]])
hp = pd.DataFrame(haq_pos)
hp.columns = ["Position", "HAQER"]
hp = hp.set_index("Position")


In [ ]:
y = pd.read_csv("HumChp_All_PhyloP.txt.gz", sep = "\t")
yh = y.set_index("Position").join(hp)
yh = yh.dropna(subset = ["HAQER"])
b = []
for i in np.unique(yh["NearestGene"]):
    b.append([i, len(np.unique(yh[yh["NearestGene"] == i]["HAQER"]))])
b = pd.DataFrame(b)
b

In [ ]:
#b.columns = ["NearestGene", "NumHAQERs"]
#b = b.set_index("NearestGene")
vh = v.set_index("NearestGene").join(b)
vh

In [ ]:
b.to_csv("NumHAQERs.csv", sep = "\t")

In [ ]:
vh["NumHAQERs"] = vh["NumHAQERs"].replace(np.nan, 0)


In [ ]:
vh[(vh["Z-score Difference"] > 1.96) & (vh["NumHAQERs"] < 2)].shape

In [ ]:
fisher_exact([[7, 5], [281, 259]])

In [ ]:
v_sigh = v_sig[v_sig["PhyloP Difference"] > 0]

In [ ]:
v_sigh[~v_sigh["NearestGene"].isin(har_genes)]

In [ ]:
y3 = y[y["PhyloP447"] > 3]
y3["Chrom"]= [x.split(":")[0] for x in y3["Position"]]
y3["Pos1"]= [int(x.split(":")[1]) - 10 for x in y3["Position"]]
y3["Pos2"]= [int(x.split(":")[1]) + 10 for x in y3["Position"]]
y3h = y3[y3["Derived"] == "H"]
y3c = y3[y3["Derived"] == "C"]
y3h[["Chrom", "Pos1", "Pos2"]].to_csv("HumanDerived_PhyloPG3.bed", sep = "\t", header = False, index = False)
y3c[["Chrom", "Pos1", "Pos2"]].to_csv("ChimpDerived_PhyloPG3.bed", sep = "\t", header = False, index = False)

In [ ]:
har_counter = Counter(har_genes)
x = []

for key in har_counter.keys():
    x.append([key, har_counter[key]])
z = pd.DataFrame(x)
z.columns = ["Gene", "NumHARs"]
z = z.set_index("Gene")
z = v.set_index("NearestGene").join(z)
z["NumHARs"] = z["NumHARs"].replace(np.nan, 0)
z = z[np.abs(z["Z-score Difference"]) > 1.96]
pearsonr(z["NumHARs"], z["Z-score Difference"])

In [ ]:
z[(z["Z-score Difference"] > 1.96) & (z["NumHARs"] >= 3)].shape

In [ ]:
fisher_exact([[32, 8], [256, 256]])

In [ ]:
y = pd.read_csv("HumChp_All_PhyloP.txt.gz", sep = "\t")
y = y[y["SpecSup447"] > 250]
y_sig = y[y["NearestGene"].isin(v_sig["NearestGene"])]


In [ ]:
har_hag = har[har["HAR_ID"].isin(HAR_IDs)]

In [ ]:
keep = []
for index, row in har_hag.iterrows():
    keep = keep + [row["chr"] + ":" + str(x) for x in range(row["start"], row["stop"] + 1)]
y_sig_har = y_sig[y_sig["Position"].isin(keep)]

In [ ]:
y_sig = y_sig[y_sig["Derived"] == "H"]
y_sig[(y_sig["PhyloP447"] > 3) & (y_sig["NearestGene"].isin(har_genes))]

In [ ]:
plt.figure(figsize=(2.5, 4), dpi=600)

sns.barplot(x = ["HAGs", "CAGs"], y = [32, 8], hue = ["HAGs", "CAGs"], palette = {"HAGs":"#E31A1C", "CAGs":"#0058FF"})
plt.ylabel("Number of genes with > 2 HARs", size = 14)
#plt.ylim(0, 555)
plt.xticks(size = 13)
plt.yticks(size = 13)
plt.title("Nominal HAGs tend to\nhave multiple HARs", size = 15)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 34, 0             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h - 1,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(2.5, 4), dpi=600)

sns.barplot(x = ["Not in HARs", "In HARs"], y = [2237 - 125, 125], hue = ["Not in HARs", "In HARs"], palette = {"Not in HARs":"magenta", "In HARs":"green"}, alpha = 0.8)
plt.ylabel("Number of subs. with PhyloP > 3", size = 14)
#plt.ylim(0, 555)
plt.xticks(size = 13)
plt.yticks(size = 13)
plt.title("HAGs near\none or more HARs", size = 15)
plt.legend([], [], frameon = False)
#x1, x2 = 0, 1                 # bar positions
#y, h = 500, 0             # height of line + vertical offset
#plt.plot([x1, x1, x2, x2],
#         [y, y + h, y + h, y],
#         lw=1.2, c='black')

#plt.text((x1 + x2) * 0.5, y + h - 5,
#         "***",
#         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
y_sig_har[(y_sig_har["PhyloP447"] > 3) & (y_sig_har["Derived"] == "H") & (y_sig_har["NearestGene"] == "RBFOX1")].shape

In [ ]:
sig = np.intersect1d(v[(v["FDR Difference"] < 0.25) & (v["FDR Difference Corr Tot"] < 0.25)]["NearestGene"], v2[(v2["FDR Difference"] < 0.25) & (v2["FDR Difference Corr Tot"] < 0.25)]["NearestGene"])
nsig = np.intersect1d(v[(v["FDR Difference"] > 0.25) & (v["FDR Difference Corr Tot"] > 0.25)]["NearestGene"], v2[(v2["FDR Difference"] > 0.05) & (v2["FDR Difference Corr Tot"] > 0.05)]["NearestGene"])
v_sig = v[v["NearestGene"].isin(sig)]
v_nsig = v[v["NearestGene"].isin(nsig)]
v2_sig = v2[v2["NearestGene"].isin(sig)]
v2_nsig = v2[v2["NearestGene"].isin(nsig)]


In [ ]:
x = pd.read_csv("../Myriad_RNA/MN_Filtered.txt", sep = "\t")
x = x[x["padj_mine Chpreffed"] < 0.05]
xs = x[x["gene"].isin(v_sig["NearestGene"])]
xns = x[x["gene"].isin(v_nsig["NearestGene"])]
xs[xs["L2FC Chpreffed"] < -0.5].shape

In [ ]:
from scipy.stats import binomtest
ha = v_sig[(v_sig["Z-score Difference Corr Tot"] > 1.96) & (v_sig["Z-score Difference"] > 1.96)].shape[0]
ca = v_sig[(v_sig["Z-score Difference Corr Tot"] < -1.96) & (v_sig["Z-score Difference"] < -1.96)].shape[0]

print(ha, ca, binomtest(ha, ha + ca))

In [ ]:
plt.figure(figsize=(6, 4), dpi=600)

sns.barplot(y=["Human", "Chimp"], x=[25, 12], hue=["Human", "Chimp"], palette=palette, alpha=0.8)

plt.xlabel("Number of genes with accelerated\n$\it{cis}$-regulatory neighborhoods", size = 13)
plt.xlim(0, 30)
plt.xticks(size=12)
plt.yticks(size=12)
plt.title("More human acceleration in non-coding DNA", size = 14)
plt.legend([], [], frameon=False)

# --- significance bracket (horizontal) ---
y1, y2 = 0, 1        # bar positions (Human, Chimp)
x, h = 27, 0.3        # x-position of bracket + horizontal offset

plt.plot([x, x + h, x + h, x], [y1, y1, y2, y2], lw=1.2, c='black')

plt.text(x + 2*h, (y1 + y2) * 0.5, "*", ha='left', va='center', fontsize=14)

plt.show()

In [ ]:
v = filter_nc("PhyloP_NC_NoFiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_Results.txt")
v2 = filter_nc("PhyloP_NC_FiltWGS_PhyloP-100_SpecSup250_YCM_NCV_Per0_PerGene_Results.txt")

sig = np.intersect1d(v[(v["FDR Difference"] < 0.05) & (v["FDR Difference Corr Tot"] < 0.05)]["NearestGene"], v2[(v2["FDR Difference"] < 0.25) & (v2["FDR Difference Corr Tot"] < 0.25)]["NearestGene"])
nsig = np.intersect1d(v[(v["FDR Difference"] > 0.05) & (v["FDR Difference Corr Tot"] > 0.05)]["NearestGene"], v2[(v2["FDR Difference"] > 0.05) & (v2["FDR Difference Corr Tot"] > 0.05)]["NearestGene"])
v_sig = v[v["NearestGene"].isin(sig)]
v_nsig = v[v["NearestGene"].isin(nsig)]
v2_sig = v2[v2["NearestGene"].isin(sig)]
v2_nsig = v2[v2["NearestGene"].isin(nsig)]

v_filt = pd.concat([v_sig, v_nsig])
v2_filt = pd.concat([v2_sig, v2_nsig])
mvp(v_filt, xlab = "Difference in PhyloP", title = "Accelerated evolution of $\it{cis}$-regulatory landscapes", fdr_col = "FDR Difference")

In [ ]:
#Doing the testing for biological process
out = []
for key in d_BP.keys():
    b = v_filt[v_filt["NearestGene"].isin(d_BP[key])]
    vnb = v_filt[~v_filt["NearestGene"].isin(d_BP[key])]
    if b.shape[0] >= 15:
        out.append([key, b.shape[0], ttest_ind(b["Z-score Difference"], vnb["Z-score Difference"])[1], ttest_ind(b["Z-score Difference"], vnb["Z-score Difference"])[0], np.mean(b["Z-score Difference"]), np.mean(vnb["Z-score Difference"])])
df = pd.DataFrame(out)
df["FDR"] = fdrcorrection(df[2])[1]
df.sort_values(2)

In [ ]:
df.columns = ["Term", "Number of genes", "p-value", "T-statistic", "Mean human z-score", "Mean chimp z-score", "FDR"]
df = df[["Term", "Number of genes", "Mean human z-score", "Mean chimp z-score", "p-value", "FDR"]]
df.to_csv("Supplemental_Tables/Supplemental_Table6.csv", index = False)

In [ ]:
fig, ax = plt.subplots(figsize = (6, 4), dpi = 450)
df = pd.DataFrame(out)
df["FDR"] = fdrcorrection(df[2])[1]

df.columns = ["Term", "Number of genes", "p-value", "T-statistic", "Mean fore", "Mean back", "FDR"]
df["Difference in means"] = df["Mean fore"] - df["Mean back"]
df = df.sort_values("p-value")
df["-log$_{10}$(FDR)"] = -np.log10(df["FDR"])
s = []
for index, row in df.iterrows():
    if row["FDR"] < 0.05:
        s.append("FDR < 0.05")
    else:
        s.append("Not significant")
df["Significance"] = s
sns.scatterplot(data = df, x = "Difference in means", y = "-log$_{10}$(FDR)", hue = "Significance", palette = {"FDR < 0.05":"magenta", "Not significant":"grey"})
plt.legend(fontsize = 14)
plt.title("GOBP $\it{cis}$-regulatory acceleration")

In [ ]:
print("Repressor", ttest_ind(br, bb))
print("Activator", ttest_ind(ba, bb))
print(np.mean(br), np.mean(ba), np.mean(bb))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=600)
ba = v_filt[v_filt["NearestGene"].isin(np.setdiff1d(pos, neg))]["Z-score Difference"]
br = v_filt[v_filt["NearestGene"].isin(np.setdiff1d(neg, pos))]["Z-score Difference"]
bb = v_filt[~v_filt["NearestGene"].isin(d_BP['Positive Regulation Of DNA-templated Transcription (GO:0045893)'] + d_BP['Negative Regulation Of DNA-templated Transcription (GO:0045892)'])]["Z-score Difference"]

palette_tr = {
    "Repressors": palette["Human"],
    "Activators": "orange",
    "Other genes": "black"
}


data = {
    #"Activators": list(ba),
    "Repressors": list(br),
    "Other genes": list(bb)
}

for label, values in data.items():
    sns.kdeplot(values, common_norm=False, fill=True, 
                color=palette_tr[label], label=label, ax=ax)

ax.legend(loc="upper right", bbox_to_anchor=(0.95, 1), fontsize = 12)
plt.xlabel("Z-score", size = 13)
plt.ylabel("Density", size = 13)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("Accelerated evolution of repressors in humans", size = 15)

In [ ]:
#ASD enrichment stuff
z = pd.read_csv("hag_cag_nag_full_annotation.csv")
b = pd.read_csv("SFARI-Gene_genes_03-28-2024release_05-09-2024export.csv")
b1 = b[b["gene-score"].isin([1, 2, 3])]


In [ ]:
#We did fisher's exact tests on all this
hag_asd = z[(z["group3"] == "HAG") & (z["gene"].isin(b1["gene-symbol"]))].sort_values("gene").shape[0]
cag_asd = z[(z["group3"] == "CAG") & (z["gene"].isin(b1["gene-symbol"]))].sort_values("gene").shape[0]
nag_asd = z[(z["group3"] == "NAG") & (z["gene"].isin(b1["gene-symbol"]))].sort_values("gene").shape[0]
hag_nasd = z[(z["group3"] == "HAG") & (~z["gene"].isin(b1["gene-symbol"]))].sort_values("gene").shape[0]
cag_nasd = z[(z["group3"] == "CAG") & (~z["gene"].isin(b1["gene-symbol"]))].sort_values("gene").shape[0]
nag_nasd = z[(z["group3"] == "NAG") & (~z["gene"].isin(b1["gene-symbol"]))].sort_values("gene").shape[0]

In [ ]:
fig, ax = plt.subplots(figsize=(2, 4), dpi=600)
sns.barplot(data = dfpp, x = "Accelerated", y = "Number SFARI genes", hue = "Accelerated", alpha = 0.85, palette = {"Human":palette["Human"], "Chimp":palette["Chimp"]})
#plt.axhline(149/(149 + 128)*(33), linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
#plt.axhline(128/(149 + 128)*(33), linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
plt.axhline(23/2.1035714285714286, linestyle = "--", color = "black")
plt.ylabel("Number of ASD-linked genes", size = 11)
plt.xlabel("", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
y, h = 25, 0 
#y, h = 21, 0 # height of line + vertical offset
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 0.5,
         "*",
         ha='center', va='bottom', fontsize=14)
plt.title("ASD-linked gene\nenrichment", size = 12)
plt.ylim(0, 28)
plt.xlabel("Lineage with acceleration")
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
z = pd.read_csv("panel_E_pct_marker_genes_top25.csv")

dfpp = pd.DataFrame([[23, 10]]).T
dfpp.columns = ["Number SFARI genes"]
dfpp["Accelerated"] = ["Human", "Chimp"]

In [ ]:
dfpp = pd.DataFrame([[25, 12]]).T
dfpp.columns = ["Number SFARI genes"]
dfpp["Accelerated"] = ["Human", "Chimp"]

from math import floor
fig, ax = plt.subplots(figsize=(4, 2), dpi=600)
sns.barplot(data = dfpp, y = "Accelerated", x = "Number SFARI genes", hue = "Accelerated", alpha = 0.85, palette = {"Human":palette["Human"], "Chimp":palette["Chimp"]})
plt.axvline(floor(25/3.1283391405342624), linestyle = "--", color = "black")
#plt.axvline(128/(148 + 128)*(37), linestyle = "--", color = "black", ymin = 0.055, ymax = 0.4535)
#plt.axvline(148/(148 + 128)*(37), linestyle = "--", color = "black", ymin = 0.5 + 0.055, ymax = 0.5 + 0.4535)
plt.xlabel("Number of marker genes in developing cortex", size = 11)
plt.ylabel("", size = 11)
plt.yticks(size = 10)
plt.xticks(size = 10)
x1, x22 = 0, 1                 # bar positions
y, h = 28, 0 
#y, h = 21, 0 # height of line + vertical offset
plt.plot([y, y + h, y + h, y], [x1, x1, x22, x22],
         lw=1.2, c='black')

plt.text(y + 0.5, (x1 + x22) * 0.5 + 0.175,
         "*",
         ha='center', va='bottom', fontsize=14)
plt.title("Cell type-specificity in developing cortex", size = 13)
plt.xlim(0, 30)
plt.ylabel("Lineage with acceleration")
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
import matplotlib.patches as mpatches


per_ct_df = pd.read_csv("hag_cag_neuronal_marker_enrichment_per_celltype.csv")
per_ct_df = per_ct_df[per_ct_df["p_adj_HAG"] < 0.05]
ct_plot = per_ct_df.sort_values('OR_HAG_vs_NAG', ascending=True).copy()
y_pos   = np.arange(len(ct_plot))

colors_bar = [
    palette["Human"] if p < 0.05 else ('#F5A07A' if p < 0.10 else '#DDDDDD')
    for p in ct_plot['p_adj_HAG']
]

fig, ax = plt.subplots(figsize=(4, 2), dpi = 450)
ax.barh(y_pos, ct_plot['OR_HAG_vs_NAG'], color=colors_bar, edgecolor='white', height=0.7, alpha = 0.85)
ax.axvline(1.0, color='black', linewidth=1.0, linestyle='--')

x_max = ct_plot['OR_HAG_vs_NAG'].max()
ax.set_xlim(0, x_max * 1.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(ct_plot['cell_type'], fontsize=11)

ax.set_xlabel('Odds ratio for top-25 marker status', fontsize=11)
ax.set_title(
    'Per-neuronal-cell-type HAG marker enrichment',
    fontsize=12
)

# Gene annotations for OR > 2
for i, (_, row) in enumerate(ct_plot.iterrows()):
    if row['OR_HAG_vs_NAG'] > 2.0 and row['top_HAG_genes']:
        ax.text(row['OR_HAG_vs_NAG'] + 0.15, i, "$\it{" + row['top_HAG_genes'] + "}$",
                va='center', fontsize=9, color='#000000')

#patches = [
#    mpatches.Patch(color=palette["Human"], label='FDR < 0.05'),
#]
#ax.legend(handles=patches, fontsize=10)
plt.xticks(size = 12)
plt.xlim(0, 22)
sns.despine(ax=ax)
#plt.tight_layout()
#plt.savefig('panel_D_only.png', dpi=150, bbox_inches='tight')
plt.show()
#print('Saved panel_D_only.png')

In [ ]:
k = []
for index, row in v_filt.iterrows():
    if row["Difference in PhyloP-weighted sum"] < 0 and row["Gene symbol"] in sig:
        k.append("Chimpanzee-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] < 0 and row["Gene symbol"] in np.setdiff1d(nominal_sig, sig):
        k.append("Nominally chimpanzee-accelerated, p < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["Gene symbol"] in sig:
        k.append("Human-accelerated, FDR < 0.05")
    elif row["Difference in PhyloP-weighted sum"] > 0 and row["Gene symbol"] in np.setdiff1d(nominal_sig, sig):
        k.append("Nominally human-accelerated, p < 0.05")
    else:
        k.append("Not accelerated")
v_filt["Acceleration"] = k
v_filt.to_csv("Supplemental_Tables/Supplemental_Table5.csv", index = False)